In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import math
from scipy.optimize import minimize

# Irradiance Data

In [ ]:
# Read the irradiance data from the CSV file
dfIrr2 = pd.read_csv('______.csv', skiprows=[1])
dfIrr2.columns = ['AbsoluteIrradiance:245', 'Unnamed: 1']

# Round the wavelengths to the nearest whole number
dfIrr2['rounded_wavelength'] = dfIrr2['AbsoluteIrradiance:245'].round()

dfIrr2.loc[dfIrr2['rounded_wavelength'] < 300, 'Unnamed: 1'] = 0

# Filter the DataFrame to include only wavelengths from 200 to 800
filtered_df1 = dfIrr2[(dfIrr2['rounded_wavelength'] >= 200) & (dfIrr2['rounded_wavelength'] <= 800)]

# Group by the rounded wavelengths and calculate the average irradiance
average_irradiance2 = filtered_df1.groupby('rounded_wavelength')['Unnamed: 1'].mean().reset_index()

# Rename the columns for clarity
average_irradiance2.columns = ['wavelength', 'average_irradiance']

Irr2=average_irradiance2['average_irradiance']
wavelengthIrr=average_irradiance2['wavelength']

Irr2=Irr2.astype(float)
wavelengthIrr=wavelengthIrr.astype(float)
I_rel2=Irr2/np.sum(Irr2)

# 2NB Actinometry

In [ ]:
#First you will need to upload your csv file to this server
df = pd.read_csv('E2NB.csv')
wavelength_2NB=df['λ (nm)']
E2NB=df['2NB E']
E2NB=E2NB[:501]
wavelength_2NB=wavelength_2NB[:501]

I_rel_sub=I_rel2[:501]
I_rel_sub=I_rel_sub[::-1]

plt.plot(wavelength_2NB,E2NB/np.sum(E2NB))
plt.plot(wavelength_2NB,I_rel_sub)

In [ ]:
times=np.array([0,20,40,60,80,120,180]) 
deg=[]
deglog=np.log([i/deg[0] for i in deg])
kobs_2NB=-(np.polyfit(times,deglog,1)[0])
#Plot the results
plt.plot(times,deglog,'bo')
plt.plot(times,times*(-kobs_2NB),'b--')

#Plot legend and axis labels
#plt.legend(fontsize=14)
plt.ylabel('ln(${[2NB]_t}$/${[2NB]_0}$)',fontsize=12)
plt.xlabel('Irradiation time (sec)',fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
#plt.savefig('2NB_decay')
plt.show()

In [ ]:
quantum2NB=0.41
def optimize_scaling_factor(quantum2NB, E2NB, I_rel_sub, known_value):
    """
    Optimize the scaling factor to match the known action_spectra value.

    Parameters:
        quantum2NB: Quantum yield.
        E2NB (pandas.Series or np.ndarray): Molar absorptivity data.
        I_rel_sub (pandas.Series or np.ndarray): Relative intensity data.
        known_value (float): The known action_spectra value to match.

    Returns:
        float: The optimal scaling factor.
    """
    # Reset index and ensure inputs are numpy arrays for better performance
    quantum2NB = quantum2NB
    E2NB = E2NB
    I_rel_sub = I_rel_sub.reset_index(drop=True).values

    # Objective function to minimize the error
    def objective(scaling_factor):
        action_spectra = 2.303 * scaling_factor * quantum2NB * E2NB * I_rel_sub
        error = np.abs(np.sum(action_spectra) - known_value)  # Minimize absolute error
        return error

    # Initial guess for the scaling factor
    initial_guess = 1.0

    # Perform the optimization
    result = minimize(objective, initial_guess, method='Nelder-Mead',tol=1e-10)

    if result.success:
        optimal_scaling_factor = result.x[0]
        
        # Calculate the optimized action_spectra with the optimal scaling factor
        optimized_action_spectra = 2.303 * optimal_scaling_factor * quantum2NB * E2NB * I_rel_sub
        optimized_value = np.sum(optimized_action_spectra)

        # Print the known and optimized values for confirmation
        print(f"Known value: {known_value}")
        print(f"Optimized action_spectra value: {optimized_value}")

        return optimal_scaling_factor
    else:
        raise ValueError("Optimization failed: " + result.message)

# Example usage:
scaling_factor = optimize_scaling_factor(quantum2NB, E2NB, I_rel_sub, kobs_2NB)
#print(f"Optimized scaling factor: {scaling_factor}")
#print(kobs_2NB)

In [ ]:
I_abs_2NB=I_rel2.reset_index(drop=True)*scaling_factor

In [ ]:
c=3*10**8 #m/s
h=6.626*10**-34 #J s
Na=6.023*10**23 #mol^-1

print('I_abs_2NB=',np.sum(I_abs_2NB.reset_index(drop=True)*h*c*Na*10**10/wavelengthIrr.reset_index(drop=True)),'W m^2')